In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('dados/inmet_limpo.csv', parse_dates=['timestamp'])

print('Shape:', df.shape)
print('Colunas:', list(df.columns))
print('\nPrimeiras linhas:')
df.head()

Shape: (8760, 5)
Colunas: ['timestamp', 'temp_ar', 'umidade_ar', 'light_hours_day', 'soil_moisture']

Primeiras linhas:


,timestamp,temp_ar,umidade_ar,light_hours_day,soil_moisture
0,2025-01-01 00:00:00,21.2,76.0,0.0,70.9
1,2025-01-01 01:00:00,21.1,79.0,0.0,68.5
2,2025-01-01 02:00:00,20.5,82.0,0.0,72.4
3,2025-01-01 03:00:00,20.7,81.0,0.0,75.7
4,2025-01-01 04:00:00,20.2,83.0,0.0,69.2


In [3]:
np.random.seed(42)

n = len(df)

# Sorteia em qual dia do ciclo cada registro está (0 a 119 dias)
dias_ciclo = np.random.randint(0, 120, n)

# Define a fase com base no dia do ciclo
fase = np.select(
    [dias_ciclo < 30,
     dias_ciclo < 60,
     dias_ciclo < 90],
    ['muda',
     'florescimento',
     'frutificacao'],
    default='colheita'
)

print('Distribuição das fases:')
fases_unicas, contagem = np.unique(fase, return_counts=True)
for f, c in zip(fases_unicas, contagem):
    print(f'  {f}: {c} ({c/n*100:.1f}%)')

Distribuição das fases:
  colheita: 2192 (25.0%)
  florescimento: 2203 (25.1%)
  frutificacao: 2124 (24.2%)
  muda: 2241 (25.6%)


In [4]:
# Nitrogênio — maior na muda, reduz na colheita (ficha técnica Embrapa/Yara)
nitrogenio = np.where(fase == 'muda',        np.random.normal(220, 25, n),
             np.where(fase == 'florescimento', np.random.normal(180, 25, n),
             np.where(fase == 'frutificacao',  np.random.normal(170, 25, n),
                                               np.random.normal(145, 20, n))))

# Fósforo — maior no florescimento e frutificação (ficha técnica Embrapa)
fosforo = np.where(fase == 'muda',          np.random.normal(45, 10, n),
          np.where(fase == 'florescimento',  np.random.normal(60, 10, n),
          np.where(fase == 'frutificacao',   np.random.normal(62, 10, n),
                                             np.random.normal(50, 10, n))))

# Potássio — maior na frutificação e colheita (ficha técnica Embrapa/Yara)
potassio = np.where(fase == 'muda',          np.random.normal(230, 40, n),
           np.where(fase == 'florescimento',  np.random.normal(265, 40, n),
           np.where(fase == 'frutificacao',   np.random.normal(320, 40, n),
                                              np.random.normal(350, 40, n))))

# Aplicar limites físicos
nitrogenio = np.clip(nitrogenio, 80, 320)
fosforo    = np.clip(fosforo, 10, 90)
potassio   = np.clip(potassio, 100, 450)

print('NPK médio por fase:')
for f in ['muda', 'florescimento', 'frutificacao', 'colheita']:
    mask = fase == f
    print(f'\n  {f}:')
    print(f'    N: {nitrogenio[mask].mean():.1f} | P: {fosforo[mask].mean():.1f} | K: {potassio[mask].mean():.1f}')

NPK médio por fase:

  muda:
    N: 219.7 | P: 45.4 | K: 229.9

  florescimento:
    N: 179.9 | P: 59.9 | K: 264.6

  frutificacao:
    N: 169.4 | P: 61.8 | K: 319.6

  colheita:
    N: 145.6 | P: 49.7 | K: 349.0


In [5]:
# pH da solução nutritiva — ideal hidropônico: 5.5-6.5 (Yara Brasil / Embrapa)
ph_solo = np.random.normal(6.0, 0.45, n)
ph_solo = np.clip(ph_solo, 4.5, 8.0)

print('pH — min:', ph_solo.min().round(2),
      '| max:', ph_solo.max().round(2),
      '| média:', ph_solo.mean().round(2))

pH — min: 4.5 | max: 7.96 | média: 6.0


In [6]:
df['fase']          = fase
df['dias_ciclo']    = dias_ciclo
df['nitrogenio']    = nitrogenio.round(1)
df['fosforo']       = fosforo.round(1)
df['potassio']      = potassio.round(1)
df['ph_solo']       = ph_solo.round(2)

print('Shape:', df.shape)
print('Colunas:', list(df.columns))
df.head()

Shape: (8760, 11)
Colunas: ['timestamp', 'temp_ar', 'umidade_ar', 'light_hours_day', 'soil_moisture', 'fase', 'dias_ciclo', 'nitrogenio', 'fosforo', 'potassio', 'ph_solo']


,timestamp,temp_ar,umidade_ar,light_hours_day,soil_moisture,fase,dias_ciclo,nitrogenio,fosforo,potassio,ph_solo
0,2025-01-01 00:00:00,21.2,76.0,0.0,70.9,colheita,102,146.4,58.8,353.9,6.22
1,2025-01-01 01:00:00,21.1,79.0,0.0,68.5,florescimento,51,185.2,48.9,266.0,6.14
2,2025-01-01 02:00:00,20.5,82.0,0.0,72.4,colheita,92,120.3,50.3,401.7,6.77
3,2025-01-01 03:00:00,20.7,81.0,0.0,75.7,muda,14,200.2,63.6,260.7,5.79
4,2025-01-01 04:00:00,20.2,83.0,0.0,69.2,colheita,106,161.1,42.2,309.3,6.46


In [7]:
classes = []
motivos = []

for i in range(len(df)):

    # --- Classe 3: PROTEGER ---
    if df['temp_ar'].iloc[i] > 38.0 or df['temp_ar'].iloc[i] < 10.0:
        classes.append(3)
        motivos.append('proteger: temperatura extrema')

    elif df['umidade_ar'].iloc[i] > 85.0:
        classes.append(3)
        motivos.append('proteger: umidade alta (risco de fungos)')

    elif df['timestamp'].dt.hour.iloc[i] == 23 and df['light_hours_day'].iloc[i] < 6.0:
        classes.append(3)
        motivos.append('proteger: luz insuficiente no dia')

    # --- Classe 2: IRRIGAR / REDUZIR ---
    elif df['soil_moisture'].iloc[i] < 55.0:
        classes.append(2)
        motivos.append('irrigar: solo seco')

    elif df['soil_moisture'].iloc[i] > 85.0:
        classes.append(2)
        motivos.append('reduzir irrigacao: solo encharcado')

    # --- Classe 1: CORRIGIR NUTRIÇÃO / pH ---
    elif df['ph_solo'].iloc[i] < 5.5 or df['ph_solo'].iloc[i] > 6.8:
        classes.append(1)
        motivos.append('corrigir pH fora da faixa ideal')

    elif df['nitrogenio'].iloc[i] < 150 or df['nitrogenio'].iloc[i] > 250:
        classes.append(1)
        motivos.append('corrigir nitrogenio fora da faixa')

    elif df['fosforo'].iloc[i] < 30 or df['fosforo'].iloc[i] > 70:
        classes.append(1)
        motivos.append('corrigir fosforo fora da faixa')

    elif df['potassio'].iloc[i] < 200 or df['potassio'].iloc[i] > 350:
        classes.append(1)
        motivos.append('corrigir potassio fora da faixa')

    # --- Classe 0: NORMAL ---
    else:
        classes.append(0)
        motivos.append('condicoes dentro do ideal')

df['action_target'] = classes
df['motivo']        = motivos

print('Distribuição das classes:')
mapa = {0: 'Normal', 1: 'Corrigir Nutrição/pH', 2: 'Irrigar/Reduzir', 3: 'Proteger'}
print(df['action_target'].value_counts().rename(mapa))
print('\nExemplo de motivos:')
print(df['motivo'].value_counts())

Distribuição das classes:
action_target
Corrigir Nutrição/pH    4243
Normal                  3132
Proteger                1141
Irrigar/Reduzir          244
Name: count, dtype: int64

Exemplo de motivos:
motivo
condicoes dentro do ideal                   3132
corrigir nitrogenio fora da faixa           1562
corrigir pH fora da faixa ideal             1195
proteger: umidade alta (risco de fungos)    1102
corrigir potassio fora da faixa              900
corrigir fosforo fora da faixa               586
irrigar: solo seco                           167
reduzir irrigacao: solo encharcado            77
proteger: temperatura extrema                 39
Name: count, dtype: int64


In [8]:
features_numericas = ['temp_ar', 'umidade_ar', 'light_hours_day',
                      'soil_moisture', 'nitrogenio', 'fosforo',
                      'potassio', 'ph_solo', 'dias_ciclo']

def aumentar(df_classe, multiplicador, seed):
    np.random.seed(seed)
    frames = [df_classe]
    for i in range(multiplicador):
        copia = df_classe.copy()
        ruido = np.random.normal(0, 0.02, copia[features_numericas].shape)
        copia[features_numericas] = copia[features_numericas] * (1 + ruido)
        copia[features_numericas] = copia[features_numericas].round(2)
        copia['timestamp'] = copia['timestamp'] + pd.Timedelta(days=365 * (i + 1))
        copia['umidade_ar']    = np.clip(copia['umidade_ar'], 0, 100)
        copia['soil_moisture'] = np.clip(copia['soil_moisture'], 15, 100)
        copia['ph_solo']       = np.clip(copia['ph_solo'], 4.5, 8.0)
        copia['dias_ciclo']    = np.clip(copia['dias_ciclo'], 0, 119).astype(int)
        frames.append(copia)
    return pd.concat(frames, ignore_index=True)

# Separar por classe
df_0 = df[df['action_target'] == 0]  # 3132
df_1 = df[df['action_target'] == 1]  # 4243
df_2 = df[df['action_target'] == 2]  # 244
df_3 = df[df['action_target'] == 3]  # 1141

# Aumentar cada classe para chegar perto de 8000
df_0_aug = aumentar(df_0, 2, seed=42)   # 3132 × 3 = 9396
df_1_aug = aumentar(df_1, 1, seed=43)   # 4243 × 2 = 8486
df_2_aug = aumentar(df_2, 32, seed=44)  # 244  × 33 = 8052
df_3_aug = aumentar(df_3, 6, seed=45)   # 1141 × 7 = 7987

df_final = pd.concat([df_0_aug, df_1_aug, df_2_aug, df_3_aug], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print('Distribuição final das classes:')
print(df_final['action_target'].value_counts().sort_index().rename(mapa))
print(f'\nShape final: {df_final.shape}')

Distribuição final das classes:
action_target
Normal                  9396
Corrigir Nutrição/pH    8486
Irrigar/Reduzir         8052
Proteger                7987
Name: count, dtype: int64

Shape final: (33921, 13)


In [9]:
df_final.to_csv('dados/dataset_tomate_final.csv', index=False)

print('Arquivo dataset_tomate_final.csv gerado com sucesso!')
print(f'\nShape final: {df_final.shape}')
print(f'Colunas: {list(df_final.columns)}')
print(f'\nPeríodo: {df_final["timestamp"].min()} → {df_final["timestamp"].max()}')
print('\nEstatísticas descritivas:')
df_final.describe().round(2)

Arquivo dataset_tomate_final.csv gerado com sucesso!

Shape final: (33921, 13)
Colunas: ['timestamp', 'temp_ar', 'umidade_ar', 'light_hours_day', 'soil_moisture', 'fase', 'dias_ciclo', 'nitrogenio', 'fosforo', 'potassio', 'ph_solo', 'action_target', 'motivo']

Período: 2025-01-01 00:00:00 → 2057-12-22 21:00:00

Estatísticas descritivas:


,timestamp,temp_ar,umidade_ar,light_hours_day,soil_moisture,dias_ciclo,nitrogenio,fosforo,potassio,ph_solo,action_target
count,33921,33921.00,33921.00,33921.00,33921.00,33921.00,33921.00,33921.00,33921.00,33921.00,33921.00
mean,2030-05-26 21:01:33.817989,21.10,67.46,4.14,70.10,58.71,179.66,54.10,290.20,6.00,1.43
min,2025-01-01 00:00:00,6.06,11.70,0.00,40.64,0.00,78.32,12.37,100.00,4.50,0.00
25%,2025-12-20 08:00:00,16.70,51.36,0.00,63.00,29.00,153.50,46.10,246.07,5.70,0.00
50%,2026-12-08 16:00:00,20.35,74.11,2.02,70.85,58.00,176.66,54.07,290.50,6.00,1.00
75%,2030-05-21 08:00:00,25.10,84.46,8.15,77.42,89.00,204.22,62.00,334.40,6.32,2.00
max,2057-12-22 21:00:00,38.03,100.00,18.45,100.00,119.00,310.89,92.61,469.14,7.96,3.00
std,NaN,5.95,20.44,4.46,11.18,34.76,35.20,11.75,59.73,0.45,1.13


In [10]:
print(df_final['action_target'].value_counts().sort_index().rename(mapa))
print(f'Shape: {df_final.shape}')

action_target
Normal                  9396
Corrigir Nutrição/pH    8486
Irrigar/Reduzir         8052
Proteger                7987
Name: count, dtype: int64
Shape: (33921, 13)
